# Lecture 4: software architecture principles

This lecture concerns software development in Python, but also in all programming languages as a side effect. 

More precisely, we are here concerned by the **good practice** required to write production ready software, or quality software. 

Up to now you have learned the basics about programming. The goal is now to apply this knowledge and skills for real world code... Well, in our example it will remain small applications, but the principles are valid for real applications including the ones for PhD students, for a single developer, or for many ones working together onto the same project for some years. 

In any kind of application the first question to ask is about its **architecture**. What do we mean by that? The architecture describes the organisation between the different classes, different levels corresponding to libraries and functionalities, grouped by topics like logging, User Interface (UI), *etc.* An application depending on the context can be built upon micro-services, onto a central bus, using distributed component, or around kernels made as Russian dolls, *etc.* 

So, the main questions are:
- how to test the application (unit test, integration test)?
- how to organise the different communications between the objects?

While these questions can have different answers, in this lecture we focus on some very **basic principles** that should simplify the coder life, whatever the final chosen architecture is. We start with the *must-be-used-else-you-are-fired* **SOLID** ones (five principles, one per letter), then the **TDD** methodology, **DRY**, **KISS**, and we conclude with **YAGNI**.

## Objectives
By the end of this lecture you can:
- state each **SOLID** principle and recognise its violation in code (a class with several reasons to change, an `isinstance` ladder, a subclass that raises "not supported", a fat interface full of `NotImplementedError`, a class that builds its own collaborators);
- refactor such code with **abstract base classes** (`ABC` + `@abstractmethod`) and **dependency injection**;
- apply this course's interface convention: an interface is named with a leading capital `I` and is **pure** (every method abstract, no implementation and no state), while a class that carries implementation is an abstract base class and keeps its plain name;
- explain why DIP makes **unit testing with mocks** possible;
- describe **TDD** as the *red → green → refactor* cycle, and what **code coverage** measures;
- apply **DRY**, **KISS** and **YAGNI**, and see when they pull against each other.

## Single responsibility principle
This pattern is the `S` into SOLID. If you take a look at Wikipedia, you will see the following simple definition coming from *Uncle Bob* (*aka* *Robert C. Martin*):
```quote
A module should be responsible to one, and only one, actor.
```

Another, perhaps more comprehensible definition is:
```quote
There should never be more than one reason for a class to change.
```

You may find different examples of class that breaks this principle. Here the one from wikipedia:
```quote
As an example, consider a module that compiles and prints a report. Imagine such a module can be changed for two reasons. First, the content of the report could change. Second, the format of the report could change. These two things change for different causes. The single-responsibility principle says that these two aspects of the problem are really two separate responsibilities, and should, therefore, be in separate classes or modules. It would be a bad design to couple two things that change for different reasons at different times.
```

Obviously here *module* should be understood as *class* or *object*, as this principle is agnostic regarding the used programming language. Let us take another example, in Python this time. 

In [ ]:
%%python
# Bad conception, break the Single Responsibility Principle

class CoffeeMaker:
    def make_black_coffee(self): ...
    def add_milk(self): ...
    def make_foam(self): ...
    def prepare_an_order(self, use_milk, do_foam): ...
    def serve(self): ...
    def take_order(self): ...
    def take_payment(self): ...

Well, no need to do programming since 53 years to understand there is a conception problem here, isn't it? 

Of course, this class has a lot of responsibilities:
- it can do different recipes of coffee, 
- it can manage the payment,
- it can serve a coffee (think about an automata)...

We should use more classes, each one with exactly one responsibility (and at least one)...
We should split it into several classes (below, *order* translates the French *commande*):

In [ ]:
%%python
# better design, following SRP (and runnable!)
from __future__ import annotations


class Coffee:
    """The product: just data."""
    def __init__(self, milk: bool = False, foam: bool = False, sugar: int = 0) -> None:
        self.milk = milk
        self.foam = foam
        self.sugar = sugar

    def __repr__(self) -> str:
        return f"Coffee(milk={self.milk}, foam={self.foam}, sugar={self.sugar})"


class Order:
    """What the customer asked for (one coffee at a time)."""
    def __init__(self, milk: bool = False, foam: bool = False, sugar: int = 0) -> None:
        self.__milk = milk
        self.__foam = foam
        self.__sugar = sugar

    @property
    def milk(self) -> bool:
        return self.__milk

    @property
    def foam(self) -> bool:
        return self.__foam

    @property
    def sugar(self) -> int:
        return self.__sugar


class Barista:
    """Prepares one coffee at a time."""
    def prepare(self, order: Order) -> Coffee:
        return Coffee(order.milk, order.foam, order.sugar)


class Waiter:
    """Takes orders and serves coffees."""
    def take_order(self) -> Order:
        return Order(milk=True, foam=True, sugar=1)     # would ask the customer / a UI

    def serve(self, coffee: Coffee) -> None:
        print(f"Here is your {coffee}")


class Cashier:
    """Takes and validates the payment."""
    def take_payment(self, order: Order) -> bool:
        return True                                      # cash, card... validated here


class CoffeeMachine:
    """Orchestrates the others: order -> payment -> preparation -> service."""
    def __init__(self, waiter: Waiter, cashier: Cashier, barista: Barista) -> None:
        self.__waiter = waiter
        self.__cashier = cashier
        self.__barista = barista

    def serve_one_customer(self) -> None:
        order = self.__waiter.take_order()
        if self.__cashier.take_payment(order):
            self.__waiter.serve(self.__barista.prepare(order))


if __name__ == "__main__":
    CoffeeMachine(Waiter(), Cashier(), Barista()).serve_one_customer()


Obviously, this example application is not finished, as we need to add many things related to the automate itself. 
But it shows what is a responsibility here:
- The `CoffeeMachine` orchestrates: it serves one customer at a time (in a real machine, in an infinite loop). It receives its collaborators in its constructor — a first taste of the `D` of SOLID.
- The `Waiter` takes one order (milk, foam, sugar, maybe size and brand...) and serves the prepared coffee. Notice that **this could be split in two!** as taking an order and serving occur at different times.
- An `Order` stores all the details about the coffee to prepare.
- A `Cashier` is responsible for getting and validating the payment.
- The `Barista` prepares one order at a time.
- A `Coffee` is an object here, but just as a simple example as for a real automate we should do it for real &#x1F601; 

You probably have understood at this point where we goes: with the Single Responsibility Principle, our classes becomes lighter, with a few methods generally except may be for some classes storing data where many getters and setters may be present (*et encore...*). 
A way to limit the effect of having a large number of very small classes is to apply the following thinking of *Uncle Bob*:
```quote
Gather together the things that change for the same reasons.
Separate those things that change for different reasons.
```
Hence, a coffee can be a single class with many property for instance...

Let us try another funny example:
```python
# Before the single responsibility principle
class Model:
  
    def pre_process(self):
        pass  

    def train(self):
        pass
  
    def evaluate(self):
        pass  
    
    def predict(self):
        pass
```

Tss, that ugly isn't it? Especially considering the following steps for the pre processing:
```python
    def pre_process(self):
        #importing data
        #converting data types
        #handling missing values
        #handling outliers
        #transforming data
```
A better solution is then the following:
```python
from abc import ABC, abstractmethod


class IPreProcess(ABC):
    @abstractmethod
    def import_data(self) -> None: ...
    @abstractmethod
    def convert_data_type(self) -> None: ...
    @abstractmethod
    def handle_missing_values(self) -> None: ...
    @abstractmethod
    def handle_outliers(self) -> None: ...
    @abstractmethod
    def transform_data(self) -> None: ...


class Train: ...
class Evaluate: ...
class Predict: ...


if __name__ == "__main__":
    try:
        IPreProcess()
    except TypeError as error:
        print("as expected:", error)
```
In this example, we are using `@abstractmethod` to say to Python that a method has no implementation, leading to an interface rather than to an implementation. **This only works if the class inherits from `ABC`**: without it, `@abstractmethod` is silently ignored and the "interface" can be instantiated. Of course, **abstract class** should be overridden in the heirs, to obtain one or more than one implementation. It is a key concept in any real-world application.

As a matter of fact, the different abstract methods used into class `IPreProcess` could and should be implemented thanks to abstract class too, as we will see it later with the `D` letter of SOLID.

Notice that the single responsibility principle is quite close of KISS. But that's another story...

## An aside: our two interface conventions

Two rules apply to every interface in this course, from here to the end of Week 2. They
are *our* conventions, not Python rules -- the standard library writes `Sequence`, not
`ISequence`, and `collections.abc` would fail both of them.

**1. An interface is named with a leading capital `I`.** `IPreProcess` just above, then
`IShape`, `INotification`, `IMovable`, `ICurrencyConverter` in the sections that follow,
and every contract of Week 2's `optlab` (`IObjective`, `ILineSearch`, `IRegularizer`,
`ILinearSolver`, ...). In a course built on SOLID you read far more class names than you
write, and the prefix answers at a glance the only question that matters: *is this a
contract I implement, or a class I use?* It answers it in an import line, in a base-class
list, in a diagram and in a stub file -- none of which show you the body.

**2. An interface is pure: it contains no implementation whatsoever.** Every method
carries `@abstractmethod` and has no body -- a docstring or `...`, nothing more. No
`__init__`, no attributes, no default that heirs may override, no convenience method
written in terms of the others. An interface states *what*, never *how*, and every line of
behaviour behind it is written by the implementer.

The second rule is what makes the first one worth having. A class that carries
implementation is an **abstract base class**, which is a perfectly good thing to write --
Labwork 3's `Employee` stores an identifier and a name, and is right to -- but it is not
an interface and it does not get the `I`. Giving it the prefix would promise a contract
and hand over a half-written class instead.

Rule 2 also protects the `I` of SOLID, the Interface Segregation Principle you will meet
in a few sections. A default implementation makes it painless to bolt one more method
onto an interface, since nobody is forced to write it -- which is exactly how interfaces
grow fat. With no defaults available, every method you add is a method every implementer
must write, and you think twice.

## Open-closed principle
The second principle of SOLID is the **open-closed principle**. While some think that *Bertrand Meyer* (who first defines it) is a schizophrenic guy, most of my different personality do not agree. 
This principle states the following:
```quote
Software entities (classes, modules, functions, etc.) should be open for extension, but closed for modification.
```
It is quite important to explain here the differences:
- A module will be said to be **open if it is still available for extension**. For example, it should be possible to add fields to the data structures it contains, or new elements to the set of functions it performs.
- A module will be said to be **closed if it is available for use by other modules**. This assumes that the module has been given a well-defined, stable description (the interface in the sense of information hiding).

The modern way to apply it is to depend on **interfaces** (abstract classes) rather than on concrete classes.

Then, this principle becomes the following:
- **Use interface and build the hierarchy onto them as inner nodes**.
- **Do as many as necessary implementations and use them as outer nodes of your class hierarchy**. 
    The implementation can be modified in the future without breaking anything (there are leaves...).

As a consequence, in modern Python a concrete leaf class can be marked with the `@final` decorator from `typing` (see [PEP 591](https://peps.python.org/pep-0591/)), so that `mypy` rejects any attempt to subclass it.

The typical violation is an `isinstance` ladder that must be edited each time a new type appears (see Labwork 4, exercise 2). Its cure is polymorphism:
```python
from abc import ABC, abstractmethod
import math


# Closed for modification: this function never changes when a shape is added
def total_area(shapes: list["IShape"]) -> float:
    return sum(shape.area() for shape in shapes)


class IShape(ABC):
    @abstractmethod
    def area(self) -> float: ...


class Square(IShape):
    def __init__(self, side: float) -> None:
        self.side = side
    def area(self) -> float:
        return self.side ** 2


class Circle(IShape):
    def __init__(self, radius: float) -> None:
        self.radius = radius
    def area(self) -> float:
        return math.pi * self.radius ** 2


# Open for extension: a new shape is a new class, nothing else is edited
class Triangle(IShape):
    def __init__(self, base: float, height: float) -> None:
        self.base = base
        self.height = height
    def area(self) -> float:
        return self.base * self.height / 2


if __name__ == "__main__":
    print(total_area([Square(2), Circle(1), Triangle(3, 4)]))
```

## Liskov Substitution Principle
This principle was first proposed by Barbara Liskov in 1987.
It is close to the *Design by Contract* pattern.

In short it can be summarized by the following rule:
```quote
Function that uses an instance of a base class A must be able to use an instance of any inherited class of A without knowing it. 
``` 

Liskov's principle defines a notion substitutability for objects, where the instances of the heirs can be used in place of the parent's instance, without altering the correctness of the function/program. 

A close cousin, which you already use daily, is **duck typing** — e.g. the `len()` function (strictly speaking this is a *protocol* rather than subclassing, but the idea of substitutability is the same). 
Indeed, this function takes as parameter an instance of a class that should be a kind of container, and returns the number of elements it contains. 
*How this function is made?* 
To work following the Liskov Substitution Principle (LSP) it relies on a specific method that should be supported by any of its parameter: the `__len__()` method. 
Let us recall the corresponding part of our previously seen `LinkedList` class:
```python
class Node:
    def __init__(self, data, next=None):
        self.data = data
        self.next = next


class LinkedList:
    def __init__(self, values=()):
        self.head = None
        for value in reversed(values):
            self.head = Node(value, self.head)

    def __len__(self) -> int:
        counter = 0
        node = self.head
        while node is not None:        # FIX: was `none`
            counter = counter + 1
            node = node.next
        return counter


print(len(LinkedList([1, 2, 3])))
```
The `len()` function works with any object having the `__len__(self)->int` method:
```python
def len(container) -> int:
    return container.__len__()
```
Somewhere, it delegates to its parameter the responsibility of the length calculation. 
If the container does not have the `__len__()` method then an exception is raised.

Below is an example of simple Python code that does not follow this principle (coming from python [tutorial web site](https://www.pythontutorial.net/python-oop/python-liskov-substitution-principle/)):
```python
from abc import ABC, abstractmethod


class INotification(ABC):
    @abstractmethod
    def notify(self, message, email):
        pass


class Email(INotification):
    def notify(self, message, email):
        print(f'Send {message} to {email}')


class SMS(INotification):
    def notify(self, message, phone):
        print(f'Send {message} to {phone}')


class Contact:
    def __init__(self, name, email, phone):
        self.name = name
        self.email = email
        self.phone = phone


class NotificationManager:
    def __init__(self, notification, contact):
        self.contact = contact
        self.notification = notification

    def send(self, message):
        if isinstance(self.notification, Email):
            self.notification.notify(message, self.contact.email)
        elif isinstance(self.notification, SMS):
            self.notification.notify(message, self.contact.phone)
        else:
            raise Exception('The notification is not supported')
```
From the first class we can see first a conception problem with notifications:
the `notify` method receive the parameter `email`, that is change to `phone` by the `SMS` subclass (which is ok for the syntax, while change appears only for semantic). 
The main problem appears after with the `NotificationManager.send(self, message)` method, that relies onto the instance of the `INotification` class received during the initialization: to work properly this method has to check the real type of the notification class, in order to send the email or the phone number... This is the breaking of LSP!

Pouah!

Ugly it is, is not it? 

To respect the LSP we have to do the following modifications:
- Remove the `email` parameter from the `INotification` class.
- Add the email or phone number to the constructor of the two `final` subclasses `SMS` and `Email`.
- Modify (simplify) the `send` method of the `NotificationManager` class.

Which gives:
```python
from abc import ABC, abstractmethod


class INotification(ABC):
    @abstractmethod
    def notify(self, message: str) -> None: ...


class Email(INotification):
    def __init__(self, email: str) -> None:
        self.email = email
    def notify(self, message: str) -> None:
        print(f'Send "{message}" to {self.email}')


class SMS(INotification):
    def __init__(self, phone: str) -> None:
        self.phone = phone
    def notify(self, message: str) -> None:
        print(f'Send "{message}" to {self.phone}')


class Contact:
    def __init__(self, name: str, email: str, phone: str) -> None:
        self.name = name
        self.email = email
        self.phone = phone


class NotificationManager:
    def __init__(self, notification: INotification) -> None:
        self.notification = notification
    def send(self, message: str) -> None:
        self.notification.notify(message)      # no isinstance: any INotification works


if __name__ == "__main__":
    contact = Contact("John Doe", "john@test.com", "(408)-888-9999")
    NotificationManager(Email(contact.email)).send("Hello John")
    NotificationManager(SMS(contact.phone)).send("Hello John")
```

The manager now works with **any** `INotification`, including ones not written yet: substitutability restored (and OCP too).

## Interface Segregation Principle
This principle can be sum up with the following sentence (from Wikipedia):
```quote
Clients should not be forced to depend upon interfaces that they do not use.
```
At first glance it seems to be something obvious, indicating that we do not really understand it...

What does it really means?

It means that interfaces should be **small and role-specific**: a class should never have to implement (or a caller depend on) methods it does not need. A symptom of a violation is an implementation that raises `NotImplementedError` or "not supported" for some methods. 
Well, with an example it will be easier to see the point.
Let us consider the following example coming from [python tutorial](https://www.pythontutorial.net/python-oop/python-interface-segregation-principle/):
```python
from abc import ABC, abstractmethod


class IVehicle(ABC):
    @abstractmethod
    def go(self):
        pass

    @abstractmethod
    def fly(self):
        pass


class Aircraft(IVehicle):
    def go(self):
        print("Taxiing")

    def fly(self):
        print("Flying")


class Car(IVehicle):
    def go(self):
        print("Going")

    def fly(self):
        raise Exception('The car cannot fly')
```
The classes `IVehicle` and `Aircraft` seems to be correct at first, because the later may fly and go on the taxiway.
But, clearly we can see that the `Car` class is weird, since we have to put an exception into the `fly()` method (considering not flying cars only...).

How to solve this misconception? 
That is simple actually: by splitting interface in different *roles*:
```python
from abc import ABC, abstractmethod


class IMovable(ABC):
    @abstractmethod
    def go(self) -> None: ...


class IFlyable(IMovable):
    @abstractmethod
    def fly(self) -> None: ...


class Car(IMovable):                 # a car only depends on what it can do
    def go(self) -> None:
        print("Going")


class Aircraft(IFlyable):
    def go(self) -> None:
        print("Taxiing")
    def fly(self) -> None:
        print("Flying")


if __name__ == "__main__":
    for vehicle in (Car(), Aircraft()):
        vehicle.go()
    Aircraft().fly()
```
Contrary to Python, some languages do not accept multiple inheritance (e.g. Java). 
In this case we have to use `interface` and not `class` to define the interfaces instead of abstract class.
In fact, the main idea of the SOLID principles is to use interfaces. Python has no `interface` keyword: we use **abstract classes**, by inheriting from `ABC` in the `abc` module (*Abstract Base Class*). *(Python also offers structural interfaces with `typing.Protocol` — the typed form of duck typing — but in this course we use `ABC`.)* 

## Dependency Inversion Principle
This principle is very simple and widely used in so many frameworks that it seems to be used even by dinosaurs.
It is a fundamental principle for testing, and since testing is absolutely required in any code (your code is ready for trash else), it should be considered as an axiom.

Dependency Inversion Principle (DIP) can be stated as follows:
```quote
Depend upon abstractions, [not] concretions.
```
Waouh, so short principle! 
But, if you take a look on Wikipedia or others pages, you will see very long pages with lot of explanation. 
We try to stay short here, so we do not add too many extra details. 

In short, DIP states:
- High-level modules should not import anything from low-level modules. Both should depend on abstractions (interfaces).
- Abstractions should not depend on details. Details (concrete implementations) should depend on abstractions.

By dictating that both high-level and low-level objects must depend on the same abstraction, this design principle inverts the way some people may think about object-oriented programming.
This is very important way to think about DIP: there are two ways, one from interface to implementation, the other considering only implementations and barely interface. Of course the second way is the *dark side of coding*, that should be banished, while the first is the *light side of coding*. The second produces code quickly than the first when we start application... but when the application grows, it is more and more difficult to modify it, and should lead in most of the case to application rewriting process. The first side needs to think more before to code, but after the beginning it allows to modify and extend an application quite simply.

You probably have already used this principle in many different *frameworks* in Java, Python, C++, C#... 
A **service** is typically the result of the DIP application. Instead of considering the service as a concrete implementation, most of the frameworks proposes some abstractions of services. These abstraction are used at different level of the application, and with **dependency injection** (DI) some service providers are added for instance into the configuration layers...

Let us take an example for DIP, again from [python tutorial](https://www.pythontutorial.net/python-oop/python-dependency-inversion-principle/). 
```python
class FXConverter:
    def convert(self, from_currency, to_currency, amount):
        print(f'{amount} {from_currency} = {amount * 1.2} {to_currency}')
        return amount * 1.2


class App:
    def start(self):
        converter = FXConverter()
        converter.convert('EUR', 'USD', 100)


if __name__ == '__main__':
    app = App()
    app.start()
```
Well, normally you should have seen the problem: `App.start()` uses directly the class `FXConverter`.
It is not that we use the `converter.convert()` method, but the fact that we build an instance of this class... 
This should not be done here, since it break the DIP (we depend on the implementation, not onto an abstraction).

To solve this dark mistake, we must use an interface, then implementations, and finally **inject** the implementation into `App`:
```python
from abc import ABC, abstractmethod


class ICurrencyConverter(ABC):
    @abstractmethod
    def convert(self, from_currency: str, to_currency: str, amount: float) -> float: ...


class FXConverter(ICurrencyConverter):
    def convert(self, from_currency: str, to_currency: str, amount: float) -> float:
        print('Converting currency using FX API')
        result = amount * 1.2                               # FIX: printed 1.2 but returned 2
        print(f'{amount} {from_currency} = {result} {to_currency}')
        return result


class App:
    def __init__(self, converter: ICurrencyConverter) -> None:
        self.converter = converter                          # injected, never created here

    def start(self) -> float:
        return self.converter.convert('EUR', 'USD', 100)


if __name__ == '__main__':
    App(FXConverter()).start()                              # wiring happens at the top level
```
As you have noticed, `App` does **not** create any converter: the instance is received in the constructor (it could be received through a setter as well), and the wiring is done once, at the top level.

Recalling the previous lecture, this is exactly what makes correct *unit testing* possible: `App` is tested with a **mock** converter, so that a bug in `FXConverter` cannot make the test of `App` fail:
```python
import unittest
from unittest.mock import Mock
from dip import App, ICurrencyConverter


class TestApp(unittest.TestCase):
    def test_start_delegates_to_the_converter(self) -> None:
        converter = Mock(spec=ICurrencyConverter)
        converter.convert.return_value = 42.0
        self.assertEqual(App(converter).start(), 42.0)
        converter.convert.assert_called_once_with('EUR', 'USD', 100)


if __name__ == "__main__":
    unittest.main()
```

## Test Driven Development
Test Driven Development (TDD) is not exactly a coding principle, but more a **coding methodology**. It states the following:
- **Red**: write a small test for the next behaviour you want — it fails, since the code does not exist yet.
- **Green**: write the *simplest* code that makes it pass.
- **Refactor**: clean the code (and the tests), keeping everything green.

Then repeat, in short cycles of a few minutes. It means that the first step in programming consists of writing the test (hence thinking about the interface first), before writing the concrete implementation. 

This methodology seems crazy for most of the baby programmers, that want to start by the concrete implementation and finish by playing on their phone instead of writing good tests...
But, let us recall that a code with no test must be stored into `/dev/null`... 
Test matters more than the concrete implementation, as if your implementation does have test or does not pass the test, then it cannot be included into the rest of the application. 

This is really the case in production, where the git/svn **continuous integration** pipeline (CI) launches the tests before to accept or reject your push. The acceptation is made using the result of the tests, of course.

May be you think that a way to pass this is to not write test? 
Ha, little player you are then :wink: 
The CI pipeline should not accept your code if no test is provided too! 
This can be checked using **code coverage**: the proportion of the code (usually lines/statements, sometimes branches) that is executed by the test suite — measured in Python with the `coverage` tool or `pytest-cov`.

Of course, code coverage assumes the tests exist, but not that they are good enough... This is the responsibility of the coder, and this is checked when you apply for a job and regularly by TCBCF, *i.e.* your supervisor (TCBCF for *Test-Checker and Bad-Coder-Firer*).

## Don't Repeat Yourself!
This principle is not part of SOLID, but is one of the very important principle you have learnt during your bachelor.

**Don't Repeat Yourself** (DRY) is quite simple to understand actually: in the words of its authors (Hunt & Thomas, *The Pragmatic Programmer*), *every piece of knowledge must have a single, unambiguous, authoritative representation within a system*. In practice, you should not write the same piece of code more than once. If it is the case, then you must put it into a function or a class. A useful rule of thumb is the **rule of three**: tolerate a duplicate once, factor it out the third time — two pieces of code that merely *look* alike are not always the same knowledge. Following SOLID, this implies to write one or some interfaces if they are missing, and at least one implementation...

To be honest, this very important principle is very difficult to follow in big application, because to avoid to repeat itself one programmer may need to add some new interface/implementation... But what happens if this already exist? What happens if another coders need the same at the same time? There are a lot of situation where the DRY principle leads us to misconception, repetition in the application, or simply led us to reinvent the wheel. 

To avoid this bad consequence, we must think about the application architecture: where the classes will go? Into which module? This is another principle actually, called **Separation of Concerns** (SoC). The same thing at a higher level...

There are different kinds of applications. The most used one consists to the Russian dolls model, with a central kernel providing some basic functionalities, onto which a first layer is added with other functionalities and so on. It is not very efficient for application made by many different programmers, actually. 

Another architecture relies on different vertical layers, where each layer contains some responsibility like UI, inputs, logging, database connexion, and so on. This can be done in the same way with Russian dolls where is layer contains different parts... 

Well, as you may have understood it the most important is then the documentation about the architecture and the communication between the programmers. With Agile methods, there is a lot of meetings (daily, weekly, start/stop...) to coordinate the different levels of information, and to exchange a lot about what everyone is doing. 

Being a good programmer is not only about technical skills, but requires others more general skills, communication among the others.

## KISS
As stated by the DRY principle discussion, a good programmer needs to work its extra skills, like communication, sociability and so on. She/he must consider the group (the other programmers, but also the customers) as very important to achieve a valuable application.

For this purpose we should follow the KISS principle. 
What is it about? No, we will not kiss each others, first because the covid, second because it is an acronym! KISS means *Keep It Simple, Stupid* (a motto attributed to the aircraft engineer Kelly Johnson).

It is very important to try to write the more simple things as possible, in other words. 
Being simple, staying clear. 
Notice that Uncle Bob (*Clean Code*) insists that functions should be **small** — hardly ever 20 lines long, ideally just a handful of lines, doing one thing.

Ouch, that's short!

In practice this implies some very simple small rules:
- Use *good identifier*, that says something to others (try on your colleagues).
- Use *small functions* (a few lines), calling other small functions.
- Use *syntactic sugar* (depends on the language).
- Stay *coherent*: use enumeration, short interfaces, to give meaning to your code. 

## YAGNI
Last but not least: **YAGNI**, *You Aren't Gonna Need It* (from Extreme Programming).

It states that you should **not implement something until it is actually needed** — no speculative feature, no "just in case" parameter, no abstraction for a future that may never come. Every line you add must be written, tested, documented and maintained; a line for an imaginary need is pure cost.

YAGNI is KISS applied to time, and it balances the other principles:
- SOLID pushes you to add interfaces; YAGNI reminds you to add them **when a second implementation or a test actually needs them**, not before.
- DRY pushes you to factor code; YAGNI (and the rule of three) reminds you not to build a generic framework out of a single use.

Remember the ISP exercise of Labwork 4: we split `ICoffeeShop` into the interfaces the shops actually need, and nothing more. That is YAGNI.

So the final word of this lecture: **write the simplest code that passes the tests, keep it clean, and add abstractions only when the code asks for them.**